# Parking Sign Detection - YOLO12 Training

Training a parking sign detector for North American street signs using YOLO12.

**Dataset:** 3,213 images, 1 class (parking_sign)
- Combined from: parking-sign-coco (Roboflow) + sf-parking-signs (Figure Eight)
- All images resized to 512x512
- Train/Val/Test split: 80/10/10

**Model:** YOLO12n (attention-centric architecture, 40.6 mAP on COCO)

**Goal:** Find optimal augmentation strategy for best detection accuracy.

## 1. Setup

In [ ]:
!pip install ultralytics -q

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import shutil

# Kaggle dataset path
DATASET_PATH = Path("/kaggle/input/unified-parking-signs")
OUTPUT_PATH = Path("/kaggle/working")
DATA_YAML = DATASET_PATH / "data.yaml"

print(f"Dataset exists: {DATASET_PATH.exists()}")
if DATASET_PATH.exists():
    print(f"Contents: {list(DATASET_PATH.iterdir())}")
    !cat {DATA_YAML}

## 2. Dataset Verification

In [ ]:
# Count images in each split
for split in ["train", "valid", "test"]:
    img_dir = DATASET_PATH / split / "images"
    label_dir = DATASET_PATH / split / "labels"
    n_images = len(list(img_dir.glob("*.jpg")))
    n_labels = len(list(label_dir.glob("*.txt")))
    print(f"{split}: {n_images} images, {n_labels} labels")

## 3. Augmentation Experiments

### Pre-applied augmentations (in dataset):
- Roboflow subset: rotation ±15°, brightness ±15%, blur 0-2.5px
- SF subset: raw images (no augmentation)

### YOLO12 augmentations to test:
- Mosaic: Combines 4 images into one
- MixUp: Blends two images
- HSV: Color space augmentation
- Geometric: Scale, translate, perspective

In [ ]:
# Common training parameters
BASE_PARAMS = {
    "data": str(DATA_YAML),
    "epochs": 100,
    "imgsz": 512,
    "batch": 16,
    "patience": 20,
    "save_period": 25,
    "project": str(OUTPUT_PATH / "runs"),
    "exist_ok": True,
    "pretrained": True,
    "verbose": True,
}

# Experiment configurations
EXPERIMENTS = {
    "exp1_baseline": {
        # Minimal augmentation
        "mosaic": 0.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "degrees": 0.0,
        "translate": 0.1,
        "scale": 0.0,
        "fliplr": 0.0,
        "flipud": 0.0,
    },
    "exp2_mosaic": {
        # Add mosaic augmentation
        "mosaic": 1.0,
        "mixup": 0.0,
        "copy_paste": 0.0,
        "degrees": 0.0,
        "translate": 0.1,
        "scale": 0.5,
        "fliplr": 0.5,
        "flipud": 0.0,
    },
    "exp3_hsv": {
        # Add HSV color augmentation
        "mosaic": 1.0,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "translate": 0.1,
        "scale": 0.5,
        "fliplr": 0.5,
    },
    "exp4_geometric": {
        # Add geometric transformations
        "mosaic": 1.0,
        "degrees": 15.0,
        "translate": 0.1,
        "scale": 0.5,
        "shear": 5.0,
        "perspective": 0.0005,
        "fliplr": 0.5,
    },
    "exp5_full": {
        # All augmentations combined
        "mosaic": 1.0,
        "mixup": 0.1,
        "copy_paste": 0.1,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "degrees": 15.0,
        "translate": 0.1,
        "scale": 0.5,
        "shear": 5.0,
        "perspective": 0.0005,
        "fliplr": 0.5,
        "flipud": 0.0,
    },
}

print(f"Configured {len(EXPERIMENTS)} experiments")

## 4. Run Training Experiments

In [ ]:
results = {}

for exp_name, aug_params in EXPERIMENTS.items():
    print(f"\n{'='*60}")
    print(f"Running {exp_name}")
    print(f"{'='*60}\n")
    
    model = YOLO("yolo12n.pt")  # YOLO12n: attention-centric, 40.6 mAP
    
    train_params = {**BASE_PARAMS, **aug_params, "name": exp_name}
    train_results = model.train(**train_params)
    
    # Store results
    results[exp_name] = {
        "mAP50": train_results.results_dict.get("metrics/mAP50(B)", 0),
        "mAP50-95": train_results.results_dict.get("metrics/mAP50-95(B)", 0),
        "precision": train_results.results_dict.get("metrics/precision(B)", 0),
        "recall": train_results.results_dict.get("metrics/recall(B)", 0),
    }
    
    print(f"\n{exp_name} Results:")
    for metric, value in results[exp_name].items():
        print(f"  {metric}: {value:.4f}")

## 5. Compare Results

In [ ]:
# Create comparison DataFrame
df = pd.DataFrame(results).T
df = df.sort_values("mAP50-95", ascending=False)

print("\n" + "="*60)
print("EXPERIMENT RESULTS COMPARISON")
print("="*60)
print(df.to_string())

# Save to CSV
df.to_csv(OUTPUT_PATH / "experiment_results.csv")
print(f"\nResults saved to {OUTPUT_PATH / 'experiment_results.csv'}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mAP comparison
ax1 = axes[0]
x = range(len(df))
width = 0.35
ax1.bar([i - width/2 for i in x], df["mAP50"], width, label="mAP50", color="steelblue")
ax1.bar([i + width/2 for i in x], df["mAP50-95"], width, label="mAP50-95", color="coral")
ax1.set_xticks(x)
ax1.set_xticklabels(df.index, rotation=45, ha="right")
ax1.set_ylabel("Score")
ax1.set_title("mAP Comparison")
ax1.legend()
ax1.set_ylim(0, 1)

# Precision/Recall comparison
ax2 = axes[1]
ax2.bar([i - width/2 for i in x], df["precision"], width, label="Precision", color="forestgreen")
ax2.bar([i + width/2 for i in x], df["recall"], width, label="Recall", color="darkorange")
ax2.set_xticks(x)
ax2.set_xticklabels(df.index, rotation=45, ha="right")
ax2.set_ylabel("Score")
ax2.set_title("Precision/Recall Comparison")
ax2.legend()
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(OUTPUT_PATH / "experiment_comparison.png", dpi=150)
plt.show()

print(f"\nChart saved to {OUTPUT_PATH / 'experiment_comparison.png'}")

## 6. Evaluate Best Model

In [ ]:
# Find best experiment
best_exp = df["mAP50-95"].idxmax()
print(f"Best experiment: {best_exp}")
print(f"mAP50-95: {df.loc[best_exp, 'mAP50-95']:.4f}")

# Load best model
best_model_path = OUTPUT_PATH / "runs" / best_exp / "weights" / "best.pt"
best_model = YOLO(best_model_path)

# Run validation on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(DATA_YAML), split="test")

print(f"\nTest Set Results:")
print(f"  mAP50: {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP50-95: {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

In [ ]:
# Show training curves
from IPython.display import Image, display

results_img = OUTPUT_PATH / "runs" / best_exp / "results.png"
if results_img.exists():
    display(Image(filename=str(results_img)))

## 7. Export Best Model

In [ ]:
# Export to ONNX for deployment
print("Exporting best model to ONNX...")
onnx_path = best_model.export(format="onnx", imgsz=512, simplify=True)
print(f"Exported to: {onnx_path}")

# Also save PyTorch format
final_model_path = OUTPUT_PATH / "parking_sign_detector.pt"
shutil.copy(best_model_path, final_model_path)
print(f"PyTorch model: {final_model_path}")

## 8. Conclusions

### Dataset
- 3,213 images with 1 class (parking_sign)
- Combined from parking-sign-coco + sf-parking-signs
- 512x512 resolution, 80/10/10 split

### Model
- **YOLO12n** - Attention-centric architecture
- 40.6 mAP on COCO (+2.1% vs YOLO11n)
- Trained with 5 augmentation strategies

### Experiments
1. **Baseline**: No runtime augmentation
2. **Mosaic**: Added mosaic augmentation
3. **HSV**: Added color space augmentation
4. **Geometric**: Added perspective transforms
5. **Full**: All augmentations combined

### Key Findings
*[Fill in after running experiments]*

- Best augmentation strategy: ...
- Final mAP50-95: ...
- Training time: ...

### Model Download
- PyTorch: `parking_sign_detector.pt`
- ONNX: `parking_sign_detector.onnx`

In [ ]:
# Print final summary
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nBest Model: {best_exp}")
print(f"mAP50-95: {df.loc[best_exp, 'mAP50-95']:.4f}")
print(f"\nOutput files:")
print(f"  - {OUTPUT_PATH / 'parking_sign_detector.pt'}")
print(f"  - {OUTPUT_PATH / 'experiment_results.csv'}")
print(f"  - {OUTPUT_PATH / 'experiment_comparison.png'}")